# FINd baseline profile

Goal: measure where the reference FINd implementation spends its time, before any optimisation.

Sections:
1. Environment snapshot (reproducibility).
2. Wall-clock latency distribution on 100 images.
3. cProfile — function-level breakdown.
4. line_profiler — hot-line analysis inside top functions.
5. memory_profiler — peak memory per call.
6. Wall-clock vs pure-compute split (I/O + decode isolation).
7. Scaling sweep — runtime vs N (log-log).
8. Extrapolation to the full 55 972-image archive.
9. Summary for the report.
10. Tail investigation — what explains p95 / median ≈ 1.7?

## 1. Environment snapshot

In [ ]:
import platform, sys, subprocess
import numpy, PIL, scipy

print(f"Python     : {sys.version.split()[0]}")
print(f"Platform   : {platform.platform()}")
print(f"Machine    : {platform.machine()}")
try:
    cpu = subprocess.check_output(['sysctl', '-n', 'machdep.cpu.brand_string']).decode().strip()
    print(f"CPU        : {cpu}")
except Exception:
    pass
print(f"numpy      : {numpy.__version__}")
print(f"Pillow     : {PIL.__version__}")
print(f"scipy      : {scipy.__version__}")

In [ ]:
import sys, os
from pathlib import Path

# === PARAMETER: change HASHER_NAME and re-run notebook for each version ===
HASHER_NAME = "optimized"  # one of: "original" | "fixed" | "optimized" | "phash" | "whash"

# === Path setup (works whether notebook is at root or in notebooks/) ===
_here = Path.cwd()
PROJECT_ROOT = _here.parent if _here.name == 'notebooks' else _here
sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)

IMG_DIR = PROJECT_ROOT / 'meme_images'
FIG_DIR = PROJECT_ROOT / 'figures' / HASHER_NAME
SUMMARIES_DIR = PROJECT_ROOT / 'summaries'
PROFILES_DIR = PROJECT_ROOT / 'profiles'
FIG_DIR.mkdir(parents=True, exist_ok=True)
SUMMARIES_DIR.mkdir(exist_ok=True)
PROFILES_DIR.mkdir(exist_ok=True)

# === Hasher registry ===
from FINd import FINDHasher
from FINd_fixed import FINDHasherFixed
from FINd_optimized import FINDHasherOptimized

import imagehash
from PIL import Image as _PILImage

class PHashAdapter:
    """Thin wrapper exposing imagehash.phash with the FINDHasher.fromFile interface,
    so profile.ipynb can benchmark it through the same code paths as our hashers."""
    def fromFile(self, path):
        return imagehash.phash(_PILImage.open(path), hash_size=16)

class WHashAdapter:
    """Same wrapper for imagehash.whash (Haar wavelet, 256-bit)."""
    def fromFile(self, path):
        return imagehash.whash(_PILImage.open(path), hash_size=16, mode='haar')

HASHER_REGISTRY = {
    "original":  FINDHasher,
    "fixed":     FINDHasherFixed,
    "optimized": FINDHasherOptimized,
    "phash":     PHashAdapter,
    "whash":     WHashAdapter,
}
HASHER_CLASS = HASHER_REGISTRY[HASHER_NAME]
print(f"Using hasher: {HASHER_NAME} -> {HASHER_CLASS.__name__}")
print(f"Outputs: figures -> {FIG_DIR.relative_to(PROJECT_ROOT)} | summary -> summaries/{HASHER_NAME}_summary.json")

# === Project helpers ===
from src.sampling import load_subset
from src.timing import (
    time_hash_series,
    time_hash_preloaded,
    measure_io_vs_compute,
    scaling_sweep,
)
from src.plots import (
    plot_latency_distribution,
    plot_time_breakdown,
    plot_scaling,
    plot_io_split,
    plot_extrapolation,
)


## 2. Wall-clock latency distribution (N = 100)

In [ ]:
hasher = HASHER_CLASS()
subset = load_subset(IMG_DIR, n=100, seed=42)

timed = time_hash_series(hasher, subset, IMG_DIR)
times = [dt for _, dt in timed]

import numpy as np
print(f"n={len(times)}  mean={np.mean(times):.3f}s  median={np.median(times):.3f}s")
print(f"std={np.std(times):.3f}s  p95={np.percentile(times, 95):.3f}s  max={max(times):.3f}s")

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8, 4))
plot_latency_distribution(times, ax)
fig.tight_layout()
fig.savefig(FIG_DIR / 'fig2_latency_distribution.png', dpi=150)
plt.show()

## 3. cProfile — function-level breakdown

Deterministic per-call attribution. Look for the function(s) with the biggest `tottime`.

In [ ]:
import cProfile, pstats, io

profiler = cProfile.Profile()
profiler.enable()
for f in subset:
    hasher.fromFile(str(IMG_DIR / f))
profiler.disable()
profiler.dump_stats(str(PROFILES_DIR / f'{HASHER_NAME}.prof'))

stats = pstats.Stats(profiler).strip_dirs().sort_stats('tottime')
buf = io.StringIO(); pstats.Stats(profiler, stream=buf).strip_dirs().sort_stats('tottime').print_stats(15)
print(buf.getvalue())

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
plot_time_breakdown(stats, ax, top=8)
fig.tight_layout()
fig.savefig(FIG_DIR / 'fig1_time_breakdown.png', dpi=150)
plt.show()

## 4. line_profiler — hot lines inside top functions

Run on a smaller subset (20 images) to keep overhead manageable.

In [ ]:
%load_ext line_profiler

In [ ]:
small = subset[:20]

def run_small():
    for f in small:
        hasher.fromFile(str(IMG_DIR / f))

if hasattr(HASHER_CLASS, 'fromImage'):
    get_ipython().run_line_magic('lprun', '-f HASHER_CLASS.fromImage run_small()')
else:
    print(f'fromImage not on {HASHER_CLASS.__name__}; skipped')

In [ ]:
_legacy, _opt = 'fillFloatLumaFromBufferImage', '_luma_from_image'
if hasattr(HASHER_CLASS, _legacy):
    get_ipython().run_line_magic('lprun', f'-f HASHER_CLASS.{_legacy} run_small()')
elif _opt and hasattr(HASHER_CLASS, _opt):
    get_ipython().run_line_magic('lprun', f'-f HASHER_CLASS.{_opt} run_small()')
else:
    print(f'{_legacy}/{_opt} not on {HASHER_CLASS.__name__}; skipped (function may have been folded into other code)')

In [ ]:
_legacy, _opt = 'boxFilter', '_box_filter'
if hasattr(HASHER_CLASS, _legacy):
    get_ipython().run_line_magic('lprun', f'-f HASHER_CLASS.{_legacy} run_small()')
elif _opt and hasattr(HASHER_CLASS, _opt):
    get_ipython().run_line_magic('lprun', f'-f HASHER_CLASS.{_opt} run_small()')
else:
    print(f'{_legacy}/{_opt} not on {HASHER_CLASS.__name__}; skipped (function may have been folded into other code)')

In [ ]:
_legacy, _opt = 'dct64To16', '_dct64_to_16'
if hasattr(HASHER_CLASS, _legacy):
    get_ipython().run_line_magic('lprun', f'-f HASHER_CLASS.{_legacy} run_small()')
elif _opt and hasattr(HASHER_CLASS, _opt):
    get_ipython().run_line_magic('lprun', f'-f HASHER_CLASS.{_opt} run_small()')
else:
    print(f'{_legacy}/{_opt} not on {HASHER_CLASS.__name__}; skipped (function may have been folded into other code)')

## 5. memory_profiler — peak memory per call

In [ ]:
%load_ext memory_profiler

In [ ]:
one = str(IMG_DIR / subset[0])
%memit hasher.fromFile(one)

In [ ]:
# Memory should NOT scale with N (buffers are reused per-call)
def hash_many(n):
    for f in subset[:n]:
        hasher.fromFile(str(IMG_DIR / f))

%memit hash_many(10)
%memit hash_many(50)
%memit hash_many(100)

## 6. Wall-clock vs pure compute (I/O + decode split)

Two measurements on the same 30 images:
- `fromFile`: includes disk I/O + JPEG decode.
- `fromImage` with pre-loaded PIL objects: pure compute only.

The difference is what parallelising file reads would save.

In [ ]:
split = measure_io_vs_compute(hasher, subset[:30], IMG_DIR)
for k, v in split.items():
    print(f'{k:>22}: {v}')

fig, ax = plt.subplots(figsize=(8, 3))
plot_io_split(split, ax)
fig.tight_layout()
fig.savefig(FIG_DIR / 'fig3_io_split.png', dpi=150)
plt.show()

## 6.5 Upper-bound benchmark — what's the irreducible cost?

Establishes a floor for what any optimised version could achieve, by
doing only the unavoidable work in numpy: open file, decode, thumbnail,
luma conversion via `convert("L")`. Skips boxFilter / decimate / DCT.

If baseline / upper-bound > 30×, the bottleneck is clearly Python overhead
(not algorithmic), and numpy vectorisation is the right call. If < 10×,
vectorisation alone won't be enough and parallelism must be in the mix.

In [ ]:
import time
from PIL import Image

# Same 100-image subset as cell 5
t0 = time.perf_counter()
for f in subset:
    img = Image.open(IMG_DIR / f)
    img.thumbnail((512, 512))
    arr = np.asarray(img.convert('L'), dtype=np.float32)
upper_bound_s = time.perf_counter() - t0

ms_per_img = upper_bound_s / len(subset) * 1000
baseline_ms = float(np.mean(times)) * 1000
ratio = baseline_ms / ms_per_img

print(f'Upper-bound (open + thumbnail + numpy luma): {ms_per_img:.1f} ms/img')
print(f'Baseline (full FINd):                         {baseline_ms:.1f} ms/img')
print(f'Theoretical headroom:                          {ratio:.1f}x')

## 7. Scaling sweep — runtime vs N

Confirm the algorithm is linear in N. A sublinear or superlinear curve would be a finding.

In [ ]:
# Use a different seed so this sweep doesn't just re-hash warm-cache files from earlier cells
large_subset = load_subset(IMG_DIR, n=500, seed=99)
sweep = scaling_sweep(hasher, large_subset, IMG_DIR, sizes=[50, 100, 200, 500])
for row in sweep:
    print(row)

fig, ax = plt.subplots(figsize=(8, 4))
plot_scaling(sweep, ax)
fig.tight_layout()
fig.savefig(FIG_DIR / 'fig5_scaling.png', dpi=150)
plt.show()

## 8. Extrapolation to the full dataset (55 972 images)

Assuming per-image cost is constant at different parallelism levels.

In [ ]:
per_image = np.mean(times)
fig, ax = plt.subplots(figsize=(8, 4))
plot_extrapolation(per_image, full_n=55972, ax=ax)
fig.tight_layout()
fig.savefig(FIG_DIR / 'fig4_extrapolation.png', dpi=150)
plt.show()

## 9. Summary for the report

Key numbers (will be pasted into `private_summative/text.txt` as the Part 1 baseline):

In [ ]:
import json

summary = {
    "hasher_name": HASHER_NAME,
    "hasher_class": HASHER_CLASS.__name__,
    "n_images": len(times),
    "mean_s": float(np.mean(times)),
    "median_s": float(np.median(times)),
    "p95_s": float(np.percentile(times, 95)),
    "max_s": float(max(times)),
    "std_s": float(np.std(times)),
    "io_split": split,
    "scaling": sweep,
    "per_image_s": float(per_image),
    "extrapolation_full_archive_hours_1x": float(per_image * 55972 / 3600),
}

print(f'{HASHER_NAME.upper()} SUMMARY')
print(f'  n_images          : {summary["n_images"]}')
print(f'  mean latency      : {summary["mean_s"]*1000:.1f} ms')
print(f'  median latency    : {summary["median_s"]*1000:.1f} ms')
print(f'  p95 latency       : {summary["p95_s"]*1000:.1f} ms')
print(f'  I/O + decode      : {split["mean_io_plus_decode"]*1000:.1f} ms ({split["io_fraction"]*100:.0f}%)')
print(f'  pure compute      : {split["mean_compute"]*1000:.1f} ms ({(1-split["io_fraction"])*100:.0f}%)')
print(f'  full 55972 @ 1x   : {summary["extrapolation_full_archive_hours_1x"]:.1f} hours single-threaded')
print(f'  full 55972 @ 8x   : {summary["extrapolation_full_archive_hours_1x"] / 8:.1f} hours (perfect 8-core scaling)')

# Persist for cross-version comparison (used by report tables and acceptance bar)
summary_path = SUMMARIES_DIR / f'{HASHER_NAME}_summary.json'
with open(summary_path, 'w') as f:
    json.dump(summary, f, indent=2, default=str)
print(f'\nWritten: {summary_path.relative_to(PROJECT_ROOT)}')


## 10. Tail investigation — why is p95 almost 2× median?

Three candidate explanations for the right tail in section 2:
- **A.** Input size (pixel count) — more pixels → more iterations in Python loops.
- **B.** File size (JPEG bytes) — affects decode time.
- **C.** System noise — other processes briefly spiking CPU.

We re-measure each of the 100 images with dimensions + file size attached, then correlate.

In [ ]:
import time
from PIL import Image

rows = []
for f in subset:
    path = IMG_DIR / f
    file_bytes = path.stat().st_size
    with Image.open(path) as im:
        w, h = im.size
    t0 = time.perf_counter()
    hasher.fromFile(str(path))
    dt = time.perf_counter() - t0
    rows.append({'file': f, 'time_s': dt, 'width': w, 'height': h, 'pixels': w * h, 'bytes': file_bytes})

times_arr = np.array([r['time_s'] for r in rows])
pixels_arr = np.array([r['pixels'] for r in rows])
bytes_arr = np.array([r['bytes'] for r in rows])

corr_pixels = np.corrcoef(times_arr, pixels_arr)[0, 1]
corr_bytes = np.corrcoef(times_arr, bytes_arr)[0, 1]

print(f'Pearson correlation time vs pixel count: {corr_pixels:+.3f}  (hypothesis A)')
print(f'Pearson correlation time vs file bytes : {corr_bytes:+.3f}  (hypothesis B)')
print()
print(f'Pixel count range: {pixels_arr.min():,} - {pixels_arr.max():,}')
print(f'File size range  : {bytes_arr.min()/1024:.1f} - {bytes_arr.max()/1024:.1f} kB')
print()
sort_idx = np.argsort(times_arr)[::-1]
print('TOP-3 slowest:')
for i in sort_idx[:3]:
    r = rows[i]
    print(f"  {r['time_s']*1000:>6.0f} ms  {r['width']}x{r['height']}  {r['bytes']/1024:.1f} kB  {r['file']}")
print('TOP-3 fastest:')
for i in sort_idx[-3:]:
    r = rows[i]
    print(f"  {r['time_s']*1000:>6.0f} ms  {r['width']}x{r['height']}  {r['bytes']/1024:.1f} kB  {r['file']}")

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.scatter(pixels_arr, times_arr * 1000, alpha=0.6, color='steelblue')
ax1.set_xlabel('pixel count (width × height)')
ax1.set_ylabel('hashing time (ms)')
ax1.set_title(f'A: time vs pixel count  (r = {corr_pixels:+.3f})')
ax1.grid(alpha=0.3)

ax2.scatter(bytes_arr / 1024, times_arr * 1000, alpha=0.6, color='coral')
ax2.set_xlabel('file size (kB)')
ax2.set_ylabel('hashing time (ms)')
ax2.set_title(f'B: time vs file bytes  (r = {corr_bytes:+.3f})')
ax2.grid(alpha=0.3)

fig.tight_layout()
fig.savefig(FIG_DIR / 'fig6_tail_investigation.png', dpi=150)
plt.show()

## 11. Export this run as HTML report

Generates `reports/{HASHER_NAME}/profile_report.html` — a self-contained
deep-dive report including all text outputs (cProfile, line_profiler, memory),
figures, and the summary section. Open in a browser, embed in PDF, or share.

Note: exports the notebook *as currently saved on disk*, so save the notebook
(Cmd+S / File → Save) before running this cell.

In [ ]:
import subprocess

report_dir = PROJECT_ROOT / 'reports' / HASHER_NAME
report_dir.mkdir(parents=True, exist_ok=True)

result = subprocess.run(
    [
        'jupyter', 'nbconvert', '--to', 'html',
        str(PROJECT_ROOT / 'notebooks' / 'profile.ipynb'),
        '--output-dir', str(report_dir),
        '--output', 'profile_report.html',
    ],
    capture_output=True, text=True,
)
if result.returncode != 0:
    print('ERROR:', result.stderr)
else:
    print(result.stdout.strip() or 'OK')
    print(f'HTML report -> {(report_dir / "profile_report.html").relative_to(PROJECT_ROOT)}')
